# Comparar estimaciones Sobol de diseño finito

**Pregunta:** ¿Qué estabilidad tienen los órdenes de efecto total Saltelli/Sobol al duplicar el diseño base?

Nivel: 3. Requisitos previos: lecciones anteriores del tema. Lectura: 30 minutos.



## Objetivos

Examinar estabilidad del orden total al duplicar base Saltelli con supuestos físicos compartidos. Acuerdo y bootstrap se calculan, sin inferirse de una tabla de sensibilidad guardada.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Importar las herramientas fijadas y resolver las entradas relativas a la carpeta del notebook.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
factor_labels = {
    "work_index": "Índice de trabajo",
    "head_grade": "Ley de cabeza",
    "liberation_size": "Tamaño de liberación",
    "floatability": "Flotabilidad",
}
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. El catálogo define mineral y circuito nominales del pórfido blando. engine/data/constants.json declara semiamplitudes uniformes independientes y tolerancias de sensibilidad. Se entregan cantidades de muestras explícitas y semilla 20261031 a los métodos de incertidumbre. No se carga contrato operativo, filtro desplegado, tabla archivada ni distribución medida. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

La comparación duplica base Saltelli de ocho a dieciséis con rangos, circuito y semilla fijos. El método calcula S1, ST y semiamplitudes en cada diseño estructurado. El acuerdo de orden proviene de los ST calculados. El hipercubo previo de 32 sorteos muestra escala de salida; no es otro estimador Sobol ni conjunto físico independiente. Rechazos y constancia detienen órdenes sin fundamento. Se conservan estimaciones negativas e intervalos amplios: coincidir en orden puntual no establece importancia robusta.

## Comparación

Saltelli et al. (2010) presentan estimadores de efectos de primer orden y orden total basados en varianza; SALib (Herman y Usher, 2017) aporta el diseño y la implementación de análisis. OreFlow documenta que su bootstrap remuestrea filas de baja discrepancia como si fueran independientes, lo que puede exagerar sus límites de incertidumbre. La corrida pequeña es una demostración reproducible del método, no un estimador novedoso ni una sensibilidad validada en terreno.

Partir del circuito de pórfido blando y su mineral nominal. El método varía propiedades minerales en rangos uniformes independientes declarados, en vez de perturbar el tratamiento operativo. Una resolución nominal establece la referencia determinista de la dispersión aleatoria. Distinguir incertidumbre de los factores seleccionados e incertidumbre de las ecuaciones: esta lección muestrea únicamente la primera.

In [ ]:
# Partir del circuito de pórfido blando y su mineral nominal.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.uncertainty import uncertainty, sensitivity, perturbed

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
print("Recuperación nominal (%):", round(nominal.metrics["recovery_pct"], 3))


Resolver el factor inferior de tamaño de liberación de 0.9, manteniendo los demás factores en uno. Este cálculo aislado explica el sentido físico antes de mezclar efectos mediante muestreo global. La liberación modifica accesibilidad mineral y separación posterior. No equivale a multiplicar recuperación final por el factor ni exige una respuesta lineal alrededor del caso nominal.

In [ ]:
# Resolver el factor inferior de tamaño de liberación de 0.9, manteniendo los demás factores en
# uno.
low_ore, low_point = perturbed(case, case.nominal, {"liberation_size": 0.9})
low = simulate(low_ore, case.plant, low_point)
print("Factor de liberación:", 0.9)
print("Recuperación condicional (%):", round(low.metrics["recovery_pct"], 3))


Resolver el factor superior de tamaño de liberación de 1.1, manteniendo los demás factores en uno. Este cálculo aislado explica el sentido físico antes de mezclar efectos mediante muestreo global. La liberación modifica accesibilidad mineral y separación posterior. No equivale a multiplicar recuperación final por el factor ni exige una respuesta lineal alrededor del caso nominal.

In [ ]:
# Resolver el factor superior de tamaño de liberación de 1.1, manteniendo los demás factores en
# uno.
high_ore, high_point = perturbed(case, case.nominal, {"liberation_size": 1.1})
high = simulate(high_ore, case.plant, high_point)
print("Factor de liberación:", 1.1)
print("Recuperación condicional (%):", round(high.metrics["recovery_pct"], 3))


Propagar 32 sorteos de hipercubo latino por el motor de producción con semilla fija. Cada factor ocupa una vez cada estrato marginal, sin eliminar incertidumbre de interacción ni hacer suficientes 32 sorteos para riesgo de colas. Inspeccionar rechazos. Los cuantiles describen estados resueltos, mientras la probabilidad de cumplir todas las restricciones usa cada sorteo y cuenta los rechazados como fallas; los resúmenes tienen denominadores distintos.

In [ ]:
# Propagar 32 sorteos de hipercubo latino por el motor de producción con semilla fija.
experiment = uncertainty(case, case.nominal, samples=32, seed=20261031)
print("Sorteos resueltos:", len(experiment["solved"]))
print("Sorteos rechazados:", experiment["refused"])
print("Semiamplitudes de factores:", experiment["inputs"])
summary = pd.DataFrame(experiment["outputs"]).T[["p05", "p50", "p95", "std"]]
display(summary.round(4))
print(
    "Probabilidad de todas las restricciones:",
    round(experiment["probabilities"]["all_constraints"], 4),
)


Graficar sorteos de recuperación y referencia nominal para mostrar la distribución sin reducirla a tres cuantiles.

In [ ]:
# Graficar sorteos de recuperación y referencia nominal para mostrar la distribución sin reducirla
# a tres cuantiles..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
recovery = np.asarray(experiment["outputs"]["recovery_pct"]["values"])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.hist(recovery, bins=8, edgecolor="white")
ax.axvline(nominal.metrics["recovery_pct"], color="darkred", label="Nominal")
ax.set(
    xlabel="Recuperación (%)",
    ylabel="Sorteos\nresueltos\n(cantidad)",
    title="Propagación de incertidumbre mineral",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)


El histograma está condicionado por los rangos minerales declarados y los estados que pudo resolver el motor. Puede ser asimétrico aunque las entradas sean uniformes simétricas, porque el circuito es no lineal y restringido. La línea nominal es una configuración determinista, no el valor esperado. Pocos intervalos gráficos no establecen probabilidades raras ni pronósticos de planta; cada intervalo debe acompañarse del conteo y los rechazos.

Calcular la descomposición Saltelli/Sobol con base N=8. El diseño requiere N por la cantidad de factores más dos evaluaciones del motor. Comparar primer orden, que aísla contribuciones individuales, y orden total, que incluye interacciones. Las semiamplitudes bootstrap describen variabilidad del estimador para este diseño. Estimaciones negativas pequeñas pueden surgir del muestreo finito y deben conservarse, sin recortarlas para presentar un orden más prolijo.

In [ ]:
# Calcular la descomposición Saltelli/Sobol con base N=8.
small_indices = sensitivity(case, case.nominal, base_samples=8, seed=20261031)
print("Filas base Sobol:", small_indices["base_samples"])
print("Evaluaciones del motor Sobol:", small_indices["evaluations"])
print("Estado Sobol:", small_indices["status"])
if small_indices["status"] != "computed":
    raise RuntimeError("El diseño de sensibilidad contiene sorteos rechazados")
effects = small_indices["indices"]["recovery_pct"]
if effects.get("constant"):
    raise RuntimeError("La recuperación es constante con la tolerancia declarada")
display(pd.DataFrame({key: effects[key] for key in ("S1", "ST", "ST_conf")}))


Calcular la descomposición Saltelli/Sobol con base N=16. El diseño requiere N por la cantidad de factores más dos evaluaciones del motor. Comparar primer orden, que aísla contribuciones individuales, y orden total, que incluye interacciones. Las semiamplitudes bootstrap describen variabilidad del estimador para este diseño. Estimaciones negativas pequeñas pueden surgir del muestreo finito y deben conservarse, sin recortarlas para presentar un orden más prolijo.

In [ ]:
# Calcular la descomposición Saltelli/Sobol con base N=16.
larger_indices = sensitivity(case, case.nominal, base_samples=16, seed=20261031)
print("Filas base Sobol:", larger_indices["base_samples"])
print("Evaluaciones del motor Sobol:", larger_indices["evaluations"])
print("Estado Sobol:", larger_indices["status"])
if larger_indices["status"] != "computed":
    raise RuntimeError("El diseño de sensibilidad contiene sorteos rechazados")
effects = larger_indices["indices"]["recovery_pct"]
if effects.get("constant"):
    raise RuntimeError("La recuperación es constante con la tolerancia declarada")
display(pd.DataFrame({key: effects[key] for key in ("S1", "ST", "ST_conf")}))


Comparar sensibilidad de orden total con ambos presupuestos, incluidas las semiamplitudes bootstrap de producción.

In [ ]:
# Comparar sensibilidad de orden total con ambos presupuestos, incluidas las semiamplitudes
# bootstrap de producción..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
names = list(larger_indices["inputs"])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for offset, record in ((-0.1, small_indices), (0.1, larger_indices)):
    effect = record["indices"]["recovery_pct"]
    values = [effect["ST"][name] for name in names]
    widths = [effect["ST_conf"][name] for name in names]
    ax.errorbar(
        values,
        np.arange(len(names)) + offset,
        xerr=widths,
        fmt="o",
        label="N=" + str(record["base_samples"]),
    )
ax.set_yticks(
    range(len(names)),
    [factor_labels.get(factor_name, factor_name) for factor_name in names],
    rotation=0,
)
ax.set(
    xlabel="Índice Sobol\ntotal (1)",
    ylabel="",
    title="Estabilidad de sensibilidad en diseño\nfinito",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


Las diferencias entre presupuestos revelan inestabilidad del diseño finito, no cambios de ecuaciones físicas. Las barras bootstrap amplias impiden ordenar con confianza cuando se superponen. No son barras de error de la planta y no contemplan correlaciones minerales omitidas ni error estructural. Estos diseños pequeños enseñan el estimador y sus límites; las afirmaciones de escala de versión requieren el estudio archivado más grande y sus propios ajustes.

El diseño Saltelli empareja matrices base y filas híbridas que reemplazan un factor cada vez. El estimador necesita su correspondencia estructurada: no son muestras intercambiables de recuperación. Un rechazo invalida la descomposición, en lugar de solo reducir el tamaño. El método también suprime órdenes de salidas constantes a su tolerancia numérica relativa; de otro modo el redondeo de una salida limitada por potencia podría producir índices vistosos para una varianza sin significado físico.

Interpretar bootstrap junto a la brecha entre efectos individuales y totales. Una brecha amplia puede reflejar interacciones y, con N pequeño, ruido del estimador. Repetir un diseño mayor con otras semillas probaría estabilidad mejor que declarar dominante la mayor estimación puntual. Los cálculos actuales N=8 y N=16 son exploratorios deliberadamente y conservan su incertidumbre visible.

## Resultados

**Respuesta:** El acuerdo de orden total entre N=8 y N=16 es <!-- result:Mismo orden total (1=sí) -->0<!-- /result --> (1=sí), con máxima semiamplitud bootstrap de N=16 <!-- result:Máxima semiamplitud de efecto total (1) -->0<!-- /result -->. Leer el orden exploratorio junto a intervalos y tabla completa de efectos. Duplicar este diseño pequeño no elimina incertidumbre estructural ni establece causalidad de planta.

Vincular acuerdo de orden total y semiamplitud calculados después de comparar diseños N8 y N16.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
answer_values = {
    "P05 de recuperación (%)": experiment["outputs"]["recovery_pct"]["p05"],
    "P95 de recuperación (%)": experiment["outputs"]["recovery_pct"]["p95"],
    "Cantidad resuelta": len(experiment["solved"]),
    "Cantidad rechazada": sum(experiment["refused"].values()),
    "Factibilidad conjunta (1)": experiment["probabilities"]["all_constraints"],
}
order_small = sorted(
    small_indices["inputs"], key=small_indices["indices"]["recovery_pct"]["ST"].get
)
order_large = sorted(
    larger_indices["inputs"], key=larger_indices["indices"]["recovery_pct"]["ST"].get
)
answer_values.update(
    {
        "Mismo orden total (1=sí)": int(order_small == order_large),
        "Máxima semiamplitud de efecto total (1)": max(
            larger_indices["indices"]["recovery_pct"]["ST_conf"].values()
        ),
    }
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

Duplicar este diseño pequeño prueba estabilidad con supuestos fijos. Acuerdo, bootstrap, rechazo o constancia limitan conjuntamente la conclusión. Dependencia geológica y error estructural quedan fuera de estas estimaciones de varianza.

## Referencias

- https://doi.org/10.1016/j.cpc.2009.09.018
- https://doi.org/10.21105/joss.00097